In [3]:
import json
import numpy as np
import pandas as pd

DATA_DIR = "sepsis_mca_dataset_v2"

# ✔ use encoded traces directly
train_traces = json.load(open(f"{DATA_DIR}/X_train_encoded.json"))
test_traces  = json.load(open(f"{DATA_DIR}/X_test_encoded.json"))

df = pd.read_csv(f"{DATA_DIR}/y_test.csv")
test_labels = df["label"].values

# ✔ load mapping only for vocab size
activity_to_id = json.load(open(f"{DATA_DIR}/activity_to_id.json"))

print("Train:", len(train_traces))
print("Test:", len(test_traces))

Train: 840
Test: 198


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

WINDOW_SIZE = 5

START_TOKEN = len(activity_to_id)
END_TOKEN   = len(activity_to_id) + 1

vocab_size = len(activity_to_id) + 2

Using device: mps


In [5]:
class WindowDataset(Dataset):
    def __init__(self, traces, window_size=5):
        self.samples = []
        self.window_size = window_size

        for trace in traces:
            padded = [START_TOKEN]*(window_size-1) + trace + [END_TOKEN]

            for i in range(window_size - 1, len(padded)):
                window = padded[i-window_size+1:i]
                target = padded[i]

                self.samples.append((window, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    x = torch.tensor([item[0] for item in batch], dtype=torch.long)
    y = torch.tensor([item[1] for item in batch], dtype=torch.long)
    return x, y

In [6]:
class DAPNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        final_out = out[:, -1, :]
        logits = self.fc(final_out)
        return logits

In [7]:
train_dataset = WindowDataset(train_traces, WINDOW_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

model = DAPNN(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10

print("Training LSTM...")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

Training LSTM...
Epoch 1/10, Loss: 351.5317
Epoch 2/10, Loss: 238.1081
Epoch 3/10, Loss: 216.0314
Epoch 4/10, Loss: 206.5225
Epoch 5/10, Loss: 201.4361
Epoch 6/10, Loss: 198.3563
Epoch 7/10, Loss: 196.2246
Epoch 8/10, Loss: 194.5557
Epoch 9/10, Loss: 193.1422
Epoch 10/10, Loss: 192.1457


In [8]:
def compute_dapnn_trace_score(trace):
    padded = [START_TOKEN]*(WINDOW_SIZE-1) + trace + [END_TOKEN]
    event_scores = []

    model.eval()

    for i in range(WINDOW_SIZE - 1, len(padded)):
        window = padded[i-WINDOW_SIZE+1:i]
        true_next = padded[i]

        x = torch.tensor(window, dtype=torch.long).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy().flatten()

        p_max = np.max(probs)
        p_true = probs[true_next]

        score = (p_max - p_true) / (p_max + 1e-8)
        event_scores.append(score)

    return np.max(event_scores) if event_scores else 0

In [9]:
print("Computing TRAIN scores...")

train_scores = np.array([
    compute_dapnn_trace_score(t) for t in train_traces
])

print("Computing TEST scores...")

test_scores = np.array([
    compute_dapnn_trace_score(t) for t in test_traces
])

Computing TRAIN scores...
Computing TEST scores...


In [10]:
threshold = np.percentile(train_scores, 70)

preds = (test_scores >= threshold).astype(int)

In [11]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(test_labels, preds))

print("\nClassification Report:")
print(classification_report(test_labels, preds))

auc = roc_auc_score(test_labels, test_scores)
print("LSTM ROC-AUC:", auc)

Confusion Matrix:
[[98 43]
 [ 8 49]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.70      0.79       141
           1       0.53      0.86      0.66        57

    accuracy                           0.74       198
   macro avg       0.73      0.78      0.73       198
weighted avg       0.81      0.74      0.75       198

LSTM ROC-AUC: 0.8113724026378002
